# Olist Brazilian E-Commerce — Business Analysis & Insights
## Notebook 02 · Findings and Recommendations

**Four business areas:** Customer Value & Retention · Seller Performance & Risk ·
Delivery & Operations Reliability · Customer Experience.

## N0. Setup & Frozen Populations
Two thresholds carried from Notebook 01: a **material** review-score gap is ≥0.30 stars, a
material rate difference is ≥2 percentage points, and a material duration difference is ≥1 day.
At ~95,000 orders almost anything is statistically significant, so significance alone is never
treated as a finding.

In [3]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

DATA_DIR = Path(r"e:\reprompt-olist\reprompt\EDA\data").resolve()
EXPORT_DIR = Path(r"e:\reprompt-olist\reprompt\EDA\tableau").resolve()
EXPORT_DIR.mkdir(exist_ok=True)

sellers    = pd.read_parquet(DATA_DIR / "seller_features.parquet")
delivery   = pd.read_parquet(DATA_DIR / "order_delivery_features.parquet")
experience = pd.read_parquet(DATA_DIR / "order_experience_features.parquet")
rfm        = pd.read_parquet(DATA_DIR / "customer_rfm_features.parquet")

# Materiality thresholds (frozen in Notebook 01)
MATERIAL_REVIEW_GAP_STARS = 0.30
MATERIAL_RATE_DIFF_PP     = 2.00
MATERIAL_LAST_MILE_DAYS   = 1.00
MATERIAL_CORRELATION_RHO  = 0.20
MATERIAL_MONETARY_RELATIVE = 0.10

# Reliable sellers: 5+ delivered orders (threshold set in NB01 S15)
sellers["is_reliable_seller"] = sellers["delivered_order_count"] >= 5

reliable_sellers   = sellers[sellers["is_reliable_seller"]]
reviewed           = experience[experience["has_review"]]
delivered          = delivery[(delivery["order_status"] == "delivered")
                              & delivery["is_delivered_on_time"].notna()]
                              
reviewed_delivered = reviewed[reviewed["lateness_bucket"] != "not delivered"]

print(f"reliable_sellers   {len(reliable_sellers):>7,} of {len(sellers):,} sellers")
print(f"reviewed           {len(reviewed):>7,} of {len(experience):,} orders")
print(f"delivered          {len(delivered):>7,} of {len(delivery):,} orders")
print(f"reviewed_delivered {len(reviewed_delivered):>7,} orders")
print(f"delivery            {len(delivery):>7,} deliveries")
print(f"rfm                {len(rfm):>7,} customers")

reliable_sellers     1,766 of 3,095 sellers
reviewed            97,917 of 98,666 orders
delivered           96,470 of 98,666 orders
reviewed_delivered  95,830 orders
delivery             98,666 deliveries
rfm                 94,989 customers


In [112]:
def compare_median_value(series_repeat, series_onetime, metric_name, series_repeat_name, series_onetime_name):
    repeat_data, onetime_data = series_repeat.dropna(), series_onetime.dropna()

    repeat_median, onetime_median = repeat_data.median(), onetime_data.median()

    absolute_diff = repeat_median - onetime_median

    relative_diff = (absolute_diff / onetime_median if onetime_median != 0 else np.nan)

    # Mann-Whitney U: suitable for comparing skewed distributions without assuming normality.
    result = stats.mannwhitneyu(repeat_data, onetime_data, alternative="two-sided")

    # Rank-biserial correlation as an interpretable effect size
    n1, n2 = len(repeat_data), len(onetime_data)

    rank_biserial = 1 - (2 * result.statistic) / (n1 * n2)

    return {
        "Metric": metric_name,
        f"{series_repeat_name} N": n1,
        f"{series_onetime_name} N": n2,
        f"{series_repeat_name} median": repeat_median,
        f"{series_onetime_name} median": onetime_median,
        "Median difference": absolute_diff,
        "Relative difference %": relative_diff * 100,
        "p-value": result.pvalue,
        "Effect size": rank_biserial,
    }


## Part D. Customer Value & Retention

**Why this comes first:** whether Olist is a retention business or an acquisition business changes
how every later finding should be read. That question is settled here.

### D1. Is this a retention business at all?

**Question:** What share of customers return, and how much of the business do they represent?
**Population:** All 94,989 customers with a realised purchase.

In [4]:
rfm.columns

Index(['customer_key', 'customer_unique_id', 'customer_state',
       'first_purchase_category', 'first_purchase_month', 'recency_days',
       'recency_score', 'frequency', 'is_repeat_buyer', 'monetary',
       'monetary_score', 'avg_order_value', 'first_order_date',
       'last_order_date', 'customer_lifespan_days'],
      dtype='str')

In [389]:
n_cust   = len(rfm)
n_repeat = int(rfm["is_repeat_buyer"].sum())
repeat   = rfm[rfm["is_repeat_buyer"]]

d1 = pd.DataFrame([
    ["Customers",      f"{n_repeat:,}", f"{n_cust:,}", f"{n_repeat/n_cust*100:.1f}%"],
    ["Orders",         f"{repeat['frequency'].sum():,}", f"{rfm['frequency'].sum():,}",
                       f"{repeat['frequency'].sum()/rfm['frequency'].sum()*100:.1f}%"],
    ["Revenue (R$)",   f"{repeat['monetary'].sum():,.0f}", f"{rfm['monetary'].sum():,.0f}",
                       f"{repeat['monetary'].sum()/rfm['monetary'].sum()*100:.1f}%"],
], columns=["Measure", "Repeat buyers", "All customers", "Repeat share"])
d1

,Measure,Repeat buyers,All customers,Repeat share
0,Customers,"2,887","94,989",3.0%
1,Orders,"6,103","98,205",6.2%
2,Revenue (R$),"890,215","15,737,668",5.7%


**Result:** Repeat buyers are **3.0% of customers (2,887)**, place **6.2% of orders**, and generate
**5.7% of revenue** (R$0.9M of R$15.7M).

**Insight — the strategic finding of this project.** Olist is structurally a one-time-purchase
marketplace. Even doubling the repeat rate would add roughly 6% to revenue. Retention is not the
growth lever here; acquisition and first-order experience are.

**Recommendation:** Do not fund a loyalty programme against this base. Direct retention spend
toward the two things that are actually reachable — a strong first-order experience, and the
categories that measurably produce returning customers (D3).

### D2. Are repeat buyers worth more?

**Question:** Do returning customers spend more, or just more often?
**Population:** All customers. `monetary` and `avg_order_value` are mathematically related and highly correlated; (NB01 - S13) they should not be presented as independent evidence. monetary is the primary customer-value metric, while AOV is used as supporting context.

In [113]:
# Compare customer value between repeat and one-time buyers.
# Median is used because monetary and avg_order_value are right-skewed.
# The relative difference is reported descriptively rather than judged against a pre-defined business threshold.

repeat_mask = rfm["is_repeat_buyer"]

customer_value_comparison = pd.DataFrame([
    compare_median_value(
        rfm.loc[repeat_mask, "monetary"],
        rfm.loc[~repeat_mask, "monetary"],
        "Lifetime spend (monetary)", "Repeat Buyers", "One-time Buyers"
    ),
    compare_median_value(
        rfm.loc[repeat_mask, "avg_order_value"],
        rfm.loc[~repeat_mask, "avg_order_value"],
        "Per-order value (avg_order_value)", "Repeat Buyers", "One-time Buyers"
    )
])

customer_value_comparison

,Metric,Repeat Buyers N,One-time Buyers N,Repeat Buyers median,One-time Buyers median,Median difference,Relative difference %,p-value,Effect size
0,Lifetime spend (monetary),2887,92102,225.55,105.63,119.92,113.528354,0.000000,-0.544102
1,Per-order value (avg_order_value),2887,92102,109.91,105.63,4.28,4.051879,0.000295,-0.039499


**Result:**

**Insight:**
Repeat buyers have a 113.5% higher median lifetime spend than one-time buyers (₹225.55 vs ₹105.63). However, their median per-order value is only 4.1% higher (₹109.91 vs ₹105.63), a small difference below the 10% practical threshold.

**Conclusion:**
The higher lifetime value of repeat buyers is primarily associated with additional purchases, not materially larger orders.

**Business implication:**
Retention efforts should focus on encouraging additional purchases/repeat behaviour, rather than assuming returning customers have significantly larger basket sizes.

### D3. Does the first purchase predict a return?

**Question:** Is repeat-buyer rate associated with the customer's first-purchase category?
**Population:** 94,957 customers with a known first-purchase category, using the NB01 grouping
(categories under 30 customers collapsed to `other`). Sparse cells make the standard chi-square
approximation unsafe, so a Monte Carlo p-value is used, as frozen in NB01.

In [59]:
cat = rfm.dropna(subset=["first_purchase_category"]).copy()

small = cat["first_purchase_category"].value_counts().loc[lambda s: s < 30].index

cat["category"] = cat["first_purchase_category"].mask(cat["first_purchase_category"].isin(small), "other")

by_cat = (cat.groupby("category")["is_repeat_buyer"]
       .agg(
           customers="size",
           repeat_buyers="sum"
       ))
by_cat["repeat_rate_pct"] = (by_cat["repeat_buyers"] / by_cat["customers"] * 100).round(2)

print(by_cat.sort_values("repeat_rate_pct", ascending=False).head(5))

# Overall repeat-buyer rate
baseline = cat["is_repeat_buyer"].mean()
print(f"\n Overall repeat rate: {baseline:.2%}")

                           customers  repeat_buyers  repeat_rate_pct
category                                                            
fashion_roupa_feminina            37              4            10.81
eletrodomesticos                 687             62             9.02
moveis_colchao_e_estofado         37              3             8.11
moveis_quarto                     89              7             7.87
fashion_roupa_masculina          106              7              6.6

 Overall repeat rate: 3.04%


In [57]:
a = cat[cat['category']=='other']
a['first_purchase_category'].value_counts().sum()
# there are 174 customers with first_purchase_category in 'other'.

np.int64(174)

In [51]:
# Monte Carlo test - If every first-purchase category actually had the same 3.04% chance of producing 
# a repeat buyer, would we still see differences between categories as large as the ones in our data?
def chi2_stat(table):
    exp = table.sum(1, keepdims=True) @ table.sum(0, keepdims=True) / table.sum()
    return ((table - exp) ** 2 / np.where(exp == 0, 1, exp)).sum()

# Observed repeat / non-repeat counts
observed = np.vstack([by_cat["repeat_buyers"], by_cat["customers"] - by_cat["repeat_buyers"]])
observed_stat = chi2_stat(observed)
print(observed_stat)

# Simulate 10,000 datasets assuming every category has the same repeat rate as the overall baseline
simulated_stats = []

for _ in range(10_000):

    simulated_repeat = RNG.binomial(
        by_cat["customers"],
        baseline
    )

    simulated_table = np.vstack([
        simulated_repeat,
        by_cat["customers"] - simulated_repeat
    ])

    simulated_stats.append(chi2_stat(simulated_table))

# Monte Carlo p-value
p_value = (sum(stat >= observed_stat for stat in simulated_stats) + 1) / (len(simulated_stats) + 1)

print(f"Monte Carlo p-value: {p_value:.4f}")

436.8164736224974
Monte Carlo p-value: 0.0001


**Result:** Repeat rates differ across first-purchase categories (Monte Carlo p < 0.001, based on
10,000 simulations).
The overall repeat-buyer rate is **3.04%**. Among the categories large enough for
individual interpretation (48 of 64 categories, with at least 165 customers), repeat
rates range from **0.78% to 9.02%**.

| Category | Customers | Repeat rate |
|---|---|---|
| eletrodomesticos | 687 | **9.02%** |
| fashion_calcados | 229 | 6.11% |
| fashion_bolsas_e_acessorios | 1,726 | 5.85% |
| … | | |
| pcs | 177 | 1.13% |
| agro_industria_e_comercio | 178 | 1.12% |
| livros_tecnicos | 255 | 0.78% |

**Insight:** **First-purchase category is associated with repeat purchasing**. Some categories have
substantially higher repeat rates than the 3.04% overall baseline, while others are
lower.
For example, `eletrodomesticos` has a **9.02%** repeat rate, while `livros_tecnicos`
has **0.78%**. This is a descriptive difference, not proof that the category itself
causes customers to return.

Category may also reflect differences in product type, price, replenishment cycle, or
customer behaviour.

**Recommendation:** Use first-purchase category as a **customer-retention segmentation signal**, rather
than assuming it is a causal driver.

Higher-repeat categories can be investigated for retention opportunities, while
lower-repeat categories can be studied to understand whether the difference is driven
by product characteristics or customer behaviour.

Do not reallocate acquisition or retention spending based on this result alone; further
business validation would be needed.

### D4. How concentrated is revenue?

**Question:** How much of revenue sits with the top customers?
**Population:** All 94,989 customers, ranked by lifetime spend into deciles.

In [69]:
ranked = rfm[["monetary"]].sort_values("monetary",ascending=False).reset_index(drop=True)
ranked["decile"] = pd.qcut(ranked.index, 10, labels=[f"D{i}" for i in range(1, 11)])

concentration = ranked.groupby("decile", observed=True).agg(
    customers=("monetary", "size"),
    revenue=("monetary", "sum"),
    median_spend=("monetary", "median")
)

concentration["revenue_share_pct"] = (concentration["revenue"] / concentration["revenue"].sum() * 100).round(1)
concentration["cumulative_pct"] = concentration["revenue_share_pct"].cumsum()
concentration.round(1)

,customers,revenue,median_spend,revenue_share_pct,cumulative_pct
decile,,,,,
D1,9499,6026376.3,472.9,38.3,38.3
D2,9499,2405873.9,247.3,15.3,53.6
D3,9499,1746230.7,182.9,11.1,64.7
D4,9499,1398393.1,146.8,8.9,73.6
D5,9499,1137829.8,118.9,7.2,80.8
D6,9498,926740.3,97.7,5.9,86.7
D7,9499,744473.0,77.7,4.7,91.4
D8,9499,596975.8,63.1,3.8,95.2
D9,9499,453846.1,47.6,2.9,98.1


**Result:** The top 10% of customers generate **38.3%** of revenue; the top 20% generate **53.6%**.
The bottom half contributes 19.2%.

**Insight:** Revenue is concentrated. Because most customers are one-time buyers, high customer monetary value is largely driven by the value of their individual purchase rather than repeated purchasing. Median spend in the top decile is R$473 against R$33 in the bottom.

**Recommendation:** Segment acquisition by expected order value rather than by expected loyalty.
Prioritize high-value customer acquisition and basket-building opportunities, while treating retention as a separate lever. Because revenue is concentrated among high-spend customers, increasing the number or value of high-value customers could materially affect revenue.

### D5 — Cohort analysis:
**Question** - Does repeat behaviour vary by acquisition cohort?

In [78]:
rfm['acq_month'] = rfm['first_purchase_month']
cohort = rfm.groupby('acq_month').agg(
    customers = ('first_purchase_month', 'count'),
    repeat_buyer = ('is_repeat_buyer', 'sum')
)
cohort['repeat_rate_pct'] = (cohort['repeat_buyer']/cohort['customers']*100).round(1)
cohort

,customers,repeat_buyer,repeat_rate_pct
acq_month,,,
2016-09,2,0,0.0
2016-10,293,12,4.1
2016-12,1,1,100.0
2017-01,752,55,7.3
2017-02,1690,70,4.1
2017-03,2571,129,5.0
2017-04,2325,106,4.6
2017-05,3541,189,5.3
2017-06,3102,167,5.4


### D6. Monetary ↔ Recency correlation

**Question** — are recent customers also high-value? — that shapes the acquisition-vs-retention story.

monetary is strongly right-skewed (confirmed in NB01 S9). Pearson assumes a linear relationship on interval-scale data. Spearman uses ranks and makes no distributional assumption. This is consistent with the rank-based methodology already established throughout NB01 and NB02.

There is essentially no meaningful relationship between how recently a customer purchased and how much they have spent historically.
Already established and proved in NB01 (S13)

In [87]:
two_purchase = rfm[(rfm["is_repeat_buyer"]) & (rfm["frequency"] == 2)].copy()

median_gap = two_purchase["customer_lifespan_days"].median()
p25 = two_purchase["customer_lifespan_days"].quantile(0.25)
p75 = two_purchase["customer_lifespan_days"].quantile(0.75)

print(f"Customers with exactly 2 purchases: {len(two_purchase):,}")
print(f"Median days to second purchase : {median_gap:.0f} days")
print(f"25th percentile                : {p25:.0f} days")
print(f"75th percentile                : {p75:.0f} days")
print()
print("Business interpretation:")
print(f"Half of customers who returned did so within {median_gap:.0f} days of their first purchase.")
print(f"25% returned within {p25:.0f} days; 75% returned within {p75:.0f} days.")
print(f"A re-engagement window of {median_gap:.0f}–{p75:.0f} days captures the bulk of returnees.")


Customers with exactly 2 purchases: 2,651
Median days to second purchase : 28 days
25th percentile                : 0 days
75th percentile                : 127 days

Business interpretation:
Half of customers who returned did so within 28 days of their first purchase.
25% returned within 0 days; 75% returned within 127 days.
A re-engagement window of 28–127 days captures the bulk of returnees.


In [396]:
rev_state = rfm.groupby("customer_state").agg(
    revenue=("monetary", "sum"),
    repeat_buyer= ("is_repeat_buyer", "mean")
)
rev_state['repeat_buyer'] = (rev_state['repeat_buyer']*100).round(1)
rev_state.sort_values(by="repeat_buyer", ascending=False)

,revenue,repeat_buyer
customer_state,,
AC,19669.70,5.2
RO,57581.05,4.3
RJ,2115825.47,3.4
MT,186104.35,3.3
SP,5881735.58,3.2
GO,340736.22,3.1
RS,877645.40,3.1
MG,1842921.66,3.0
ES,323225.23,3.0


## Part A. Seller Performance & Risk

Olist carries the reputational risk of every seller on the platform. This section identifies where
that risk actually sits — and, importantly, where it only appears to.

All rate-based conclusions use **reliable sellers** (5+ delivered orders, n=1,766), per NB01 S15.

In [95]:
print(sellers.shape)
print(sellers.columns)

(3095, 34)
Index(['seller_key', 'seller_id', 'seller_city', 'seller_state',
       'seller_tenure_bucket', 'seller_volume_tier', 'total_orders',
       'total_items', 'total_revenue', 'total_freight', 'avg_order_value',
       'avg_freight_to_price_ratio', 'primary_category',
       'distinct_categories_sold', 'review_count', 'avg_review_score',
       'pct_negative_reviews', 'delivered_order_count', 'late_delivery_count',
       'late_delivery_rate', 'shipping_anomaly_count',
       'item_level_anomaly_rate', 'order_level_anomalous_order_count',
       'order_level_anomaly_rate', 'delivered_missing_approval_orders',
       'delivered_missing_multiple_dates_orders',
       'carrier_before_approval_orders', 'delivery_before_carrier_orders',
       'canceled_with_delivery_date_orders',
       'delivered_missing_final_date_orders',
       'delivered_missing_carrier_anomaly_orders',
       'has_sufficient_sample_size', 'has_sufficient_delivery_sample_size',
       'is_reliable_seller'],
  

In [94]:
reliable_sellers.shape

(1766, 34)

### A1. How exposed is Olist to a small number of sellers, and does seller volume concentration translate into similar revenue concentration?

**Question:** How concentrated is fulfilment volume? Is our fulfilment volume dependent on a relatively small number of sellers? "Does revenue concentration mirror delivered-volume concentration among the highest-volume sellers?"
**Population:** All 3,095 sellers.

In [403]:
seller_ranked = sellers.sort_values("delivered_order_count",ascending=False).copy()

seller_ranked["cum_volume_pct"] = (seller_ranked["delivered_order_count"].cumsum()
                                   / seller_ranked["delivered_order_count"].sum()* 100)

seller_ranked["cum_revenue_pct"] = (seller_ranked["total_revenue"].cumsum()
    / seller_ranked["total_revenue"].sum()* 100)

checkpoints = [1, 5, 10, 20, 50]

rows = []

for p in checkpoints:
    n = int(len(seller_ranked) * p / 100)
    row = seller_ranked.iloc[n - 1]
    top_sellers = seller_ranked.head(n)

    rows.append({
        "Top % of sellers": f"{p}%",
        "Sellers": n,
        "Share of delivered volume": f"{row['cum_volume_pct']:.1f}%",
        "Share of revenue generated": f"{row['cum_revenue_pct']:.1f}%",
        "Difference between volume & revenue":
            f"{row['cum_volume_pct'] - row['cum_revenue_pct']:.1f}%",
        "New Sellers" : (top_sellers["seller_tenure_bucket"] == "new").sum(),
        "Established Sellers": (top_sellers["seller_tenure_bucket"] == "established").sum()
    })

r1 = pd.DataFrame(rows)

r1

,Top % of sellers,Sellers,Share of delivered volume,Share of revenue generated,Difference between volume & revenue,New Sellers,Established Sellers
0,1%,30,26.4%,20.2%,6.2%,0,30
1,5%,154,53.7%,45.9%,7.8%,0,154
2,10%,309,68.7%,59.2%,9.5%,0,309
3,20%,619,83.2%,76.3%,6.9%,2,617
4,50%,1547,96.4%,93.3%,3.0%,24,1523


In [405]:
sellers["delivered_order_count"].median()

np.float64(6.0)

**Result:** 10% of sellers handle 68.7% of delivered volume but generate 59.2% of revenue. That's a meaningful difference.
Similarly: Top 20% → 83.2% of volume vs 76.3% of revenue. So revenue is consistently lower than volume share for the highest-volume seller groups. The highest-volume sellers are overwhelmingly **established sellers**. There are no new sellers in the top 1%, 5%, or 10%. Only 2 appear in the top 20%, and 24 in the top 50%.
**The highest-volume sellers handle a larger share of orders than their corresponding share of revenue.**

**Insight:** Fulfilment volume is highly concentrated among a relatively small group of sellers. This creates operational concentration risk because issues affecting major sellers could affect a large share of total volume.

**Recommendation:** Prioritise monitoring and contingency planning for high-volume sellers, while using S15's reliability filter when evaluating seller performance rates.

### A2 & A3. Is seller risk one signal or two — and does reliability filtering change the answer?

**Question:** Do sellers with more order anomalies also tend to have higher late-delivery rates?
**Population:** All sellers vs. reliable sellers only.
**Method:** Spearman correlation (both variables are zero-inflated and skewed). Material threshold: |ρ| ≥ 0.20.

In [100]:
# All sellers
all_data = sellers[["order_level_anomaly_rate", "late_delivery_rate"]].dropna()
rho_all = stats.spearmanr(
    all_data["order_level_anomaly_rate"],
    all_data["late_delivery_rate"]
).statistic

# Reliable sellers
rel_data = reliable_sellers[["order_level_anomaly_rate", "late_delivery_rate"]].dropna()
rho_rel = stats.spearmanr(
    rel_data["order_level_anomaly_rate"],
    rel_data["late_delivery_rate"]
).statistic

# Final result
result = pd.DataFrame({
    "Population": ["All sellers", "Reliable sellers only"],
    "n": [len(all_data), len(rel_data)],
    "Spearman ρ": [round(rho_all, 3), round(rho_rel, 3)]
})

result["Material?"] = result["Spearman ρ"].abs() >= MATERIAL_CORRELATION_RHO

result

,Population,n,Spearman ρ,Material?
0,All sellers,2970,0.125,False
1,Reliable sellers only,1766,-0.042,False


**Result:** ρ = The relationship is weak in the **full seller population (ρ = 0.125)** and effectively disappears among **reliable sellers (ρ = −0.042)**. Neither reaches
the 0.20 materiality threshold.

**Insight:** The weak positive correlation in the full population does not persist after restricting the analysis to sellers with sufficient delivered-order volume — a seller with 2 delivered orders and 1 late one shows a 50% late rate, which is noise rather than risk. Anomaly rate and late-delivery rate behave as separate seller-risk signals in this dataset.

This means a seller with a high anomaly rate does not necessarily have a high late-delivery rate.

**Recommendation:** Monitor the two measures separately. Anomaly rate can indicate data/process issues, while late-delivery rate reflects operational performance. Combining them into a single risk measure would make the interpretation less clear.

### A4. Do new sellers carry more risk? *(exploratory)*

**Question:** Do sellers in their first 90 days show higher anomaly rates?

**Population:** Reliable sellers, split by tenure.

**Caveat, established before running:** the reliability filter removes almost all new sellers by
construction — they have not accumulated volume yet. This comparison is **exploratory only**.

In [118]:
tenure_tbl = sellers.groupby("seller_tenure_bucket").agg(
    sellers = ("seller_key", "size"),
    median_anomaly_rate = ("order_level_anomaly_rate", "median"),
    median_late_rate=("late_delivery_rate", "median")
).reset_index()

tenure_tbl

,seller_tenure_bucket,sellers,median_anomaly_rate,median_late_rate
0,established,2850,0.0,0.0
1,new,245,0.0,0.0


### A5 — How concentrated is Olist's revenue among its sellers?

The highest-revenue sellers account for what share of revenue and volume?

In [398]:
rev = sellers["total_revenue"].sort_values(ascending=False).to_numpy()
cum_rev = rev.cumsum()/rev.sum()*100

r1 = pd.DataFrame([
    {"Top % of sellers": f"{p}%", 
     "Sellers": k,
     "Share of revenue generated": f"{cum_rev[k-1]:.1f}%"}
    for p in [1, 5, 10, 20, 50]
    for k in [int(len(rev)*p/100)]
])
r1

,Top % of sellers,Sellers,Share of revenue generated
0,1%,30,25.7%
1,5%,154,53.2%
2,10%,309,67.5%
3,20%,619,82.7%
4,50%,1547,96.8%


**Result** : Revenue is highly concentrated among a relatively small number of sellers.

**Insight** : Olist's seller revenue is highly concentrated: the top 20% of sellers by revenue generate 82.7% of total revenue.
This tells management that financial exposure is concentrated among a relatively small seller group.

Prioritize monitoring and relationship management for the highest-revenue sellers because a relatively small seller group contributes a large share of platform revenue.

### A6 — Do sellers with poor review scores overlap with sellers who deliver late?
**Question**: does poor customer satisfaction (review) come from the same sellers as poor delivery?
As seller late-delivery rate increases, does seller review score tend to decrease?

In [412]:
data = reliable_sellers[["late_delivery_rate", "avg_review_score", "pct_negative_reviews"]].dropna()

rho = stats.spearmanr(data["late_delivery_rate"], data["avg_review_score"]).statistic
rho_2 = stats.spearmanr(data["late_delivery_rate"], data["pct_negative_reviews"]).statistic

print(f"Reliable sellers: {len(data):,}")
print(f"Spearman rho (review score): {rho:.2f}")
print(f"Spearman rho (negative reviews): {rho_2:.2f}")
print(f"Material? {abs(rho) >= MATERIAL_CORRELATION_RHO}")

if abs(rho) < MATERIAL_CORRELATION_RHO:
    print("No meaningful association between seller lateness and review score.")
elif rho < 0:
    print("Higher seller lateness is associated with lower review scores.")
else:
    print("Higher seller lateness is associated with higher review scores.")

Reliable sellers: 1,766
Spearman rho (review score): -0.37
Spearman rho (negative reviews): 0.36
Material? True
Higher seller lateness is associated with lower review scores.


**Business result**: Sellers with higher late-delivery rates tend to have lower customer review scores. The relationship is not merely a small statistical fluctuation; **ρ = −0.37** indicates a moderate negative association. Because this is correlation, we should say “associated with”, not “late delivery causes poor reviews.”

**Insight**: This is useful because it connects operational performance with customer experience: delivery reliability appears to be an important part of seller-level customer satisfaction.
It also complements A2/A3:
**Anomaly rate ↔ late delivery**: no meaningful relationship.
**Late delivery ↔ review score**: meaningful relationship.

So the evidence points to late delivery being more directly connected to customer experience than the data-quality/anomaly rate.

**Recommendation**: Prioritize high-lateness sellers for delivery-performance investigation and improvement, particularly where both lateness and poor reviews occur.

### A7 - Is high freight burden primarily associated with seller category or seller geography?
Because a high freight ratio could arise from:

the type/value of products the seller sells, or the location/logistics geography of the seller.
**Population**: Reliable sellers.

**Primary metric**: Median avg_freight_to_price_ratio.

**Interpretation rule**: Only groups with at least 5 reliable sellers are individually interpreted.

In [168]:
cat_frt = reliable_sellers.groupby("primary_category").agg(
    sellers = ("seller_key", "size"),
    median_freight_ratio = ("avg_freight_to_price_ratio", "median"),
    median_orders = ("delivered_order_count", "median")
).query("sellers >= 5").sort_values(by="median_freight_ratio", ascending=False)
cat_frt.head(5)

,sellers,median_freight_ratio,median_orders
primary_category,,,
construcao_ferramentas_jardim,6,0.46630,17.5
eletronicos,25,0.43950,17.0
alimentos_bebidas,10,0.40610,9.0
livros_interesse_geral,22,0.39015,11.0
moveis_cozinha_area_de_servico_jantar_e_jardim,7,0.36460,10.0


In [170]:
state_freight = (
    reliable_sellers
    .groupby("seller_state")
    .agg(
        sellers=("seller_key", "size"),
        median_freight_ratio=("avg_freight_to_price_ratio", "median"),
        median_orders=("delivered_order_count", "median")
    ).query("sellers>=5").sort_values("median_freight_ratio", ascending=False)
)

state_freight.head(5)

,sellers,median_freight_ratio,median_orders
seller_state,,,
ES,12,0.3961,6.0
GO,20,0.3415,11.5
CE,5,0.3350,14.0
DF,22,0.2945,15.5
RS,59,0.2871,13.0


**Result**: Freight burden varies across both categories and seller states. The highest median category-level ratios were 46.6% for construction tools/garden, 44.0% for electronics, and 40.6% for food & beverages. At the state level, ES had the highest median ratio at 39.6%, followed by GO at 34.2%.

**Insight**: Freight burden appears to vary by both what sellers sell and where they operate. The results therefore do not support treating freight burden as a single seller-level issue.

The analysis is descriptive and does not establish whether category or geography is the primary driver.

**Recommendation**: Investigate high-freight categories and states separately to determine whether elevated freight is driven more by product characteristics or geographic logistics. For seller-level intervention, consider freight burden alongside seller volume rather than using the ratio alone.

## Part B. Delivery & Operations Reliability

**Population:** 96,470 delivered orders with a known outcome. Durations are heavily skewed, so
medians and p95 are reported, never means (NB01 S12).

In [171]:
delivered.columns

Index(['order_id', 'order_status', 'is_delivered', 'customer_key',
       'customer_state', 'customer_city', 'primary_seller_key',
       'primary_seller_state', 'is_cross_state', 'order_purchase_date_key',
       'approval_to_carrier_hours', 'carrier_to_customer_days',
       'carrier_to_customer_hours', 'is_delivered_on_time',
       'delivery_estimate_error_days', 'order_total_revenue',
       'order_total_freight', 'freight_to_revenue_ratio', 'primary_category',
       'has_delivery_timestamp_anomaly'],
      dtype='str')

### B1. Are we keeping the delivery promise?

**Question:** How often do we hit the promised date, and by how much do we miss when we don't?

`delivery_estimate_error_days` is defined as actual delivery date minus estimated delivery date; positive values indicate late delivery and negative values indicate early delivery.

In [173]:
on_time_rate = delivered["is_delivered_on_time"].mean()*100
err = delivered["delivery_estimate_error_days"]
late_err = err[err > 0]     # late delivery

pd.DataFrame([
    ["On-time rate", f"{on_time_rate:.1f}%"],
    ["Late orders", f"{(delivered["is_delivered_on_time"]==False).sum():,}"],
    ["Median days vs promise (all)", f"{err.median():.0f} days early"],
    ["Median days late (late orders only)", f"{late_err.median():.0f} days"],
    ["p90 days late", f"{late_err.quantile(0.90):.0f} days"],
], columns=["Measure", "Value"])

,Measure,Value
0,On-time rate,91.9%
1,Late orders,"7,826"
2,Median days vs promise (all),-12 days early
3,Median days late (late orders only),7 days
4,p90 days late,22 days


**Insight:** Olist meets the promised delivery date for **91.9%** of orders with a known outcome, but the delivery promise is conservative: the median order arrives **12 days before the promised date**.

The remaining late orders form a smaller but materially severe tail. When the promise is missed, the median order is **7 days late**, while the worst 10% of late orders are **22+ days late**.

Therefore, the overall on-time rate alone does not fully describe delivery performance. It needs to be interpreted alongside **promise calibration and late-miss severity**.


**Recommendation:**:Use the on-time rate as the headline service-level measure, but report **median and P90 days late** alongside it.

The large gap between promised and actual delivery dates also warrants investigation of **estimate calibration**. The current analysis establishes that the estimates are conservative; it does not establish whether this affects customer conversion or other commercial outcomes.

### B2. Which stage is the bottleneck?

**Question:** Is the delay on the seller's side or in the carrier network?
**Note:** the two stage durations are stored in different units (hours vs days) — normalised to days
before comparing.

For a comparison, using the same eligible orders is stronger than independently calculating each stage from potentially different populations.

In [189]:
# For the same orders, which stage takes longer? we will drop nulls 
# for the orders which have nulls in any of the both columns
stage_data = delivered[["approval_to_carrier_hours", "carrier_to_customer_days"]].dropna()

seller_side = (stage_data["approval_to_carrier_hours"] / 24)
carrier_side = stage_data["carrier_to_customer_days"]

In [190]:
pd.DataFrame([
    ["Seller -> carrier (handoff)", round(seller_side.median(), 2), round(seller_side.quantile(0.95),2)],
    ["Carrier -> customer (last mile)", round(carrier_side.median(), 2), round(carrier_side.quantile(0.95),2)],
], columns=["Stage", "Median days", "p95 days"])

,Stage,Median days,p95 days
0,Seller -> carrier (handoff),1.79,8.08
1,Carrier -> customer (last mile),7.00,24.00


**Result:** Sellers hand off in a median of **1.8 days**; the last mile takes **7.0 days** — roughly
**4× longer**. At p95 the gap is 8.1 days vs 24.0 days.

**Insight**: The **last-mile stage is the dominant delivery-time component**. Seller-to-carrier handoff takes a median of **1.79 days**, compared with **7.00 days** for carrier-to-customer delivery.

The difference persists in the upper tail: P95 duration is **8.08 days** for seller handoff versus **24.00 days** for last mile.

This indicates that the largest opportunity for reducing overall delivery elapsed time lies in understanding the **last-mile stage**, although the analysis does not establish that the carrier is causally responsible for all of the delay.

**Recommendation**: Prioritize further investigation of the **last-mile stage**, including geographic and cross-state differences, while continuing to monitor sellers with unusually long handoff times.

The next analysis should determine **where last-mile duration increases and whether cross-state or geographic patterns explain part of the gap**.

### B3. Does crossing state lines cost us?

**Question:** Are cross-state deliveries slower, and by enough to matter?
**Population:** Delivered orders with a known last-mile duration. Material difference: ≥1 day.

In [ ]:
sub = delivered.dropna(subset=["carrier_to_customer_days"])
cross_tbl = pd.DataFrame([compare_median_value(
    sub.loc[sub["is_cross_state"], "carrier_to_customer_days"],
    sub.loc[~sub["is_cross_state"], "carrier_to_customer_days"],
    "Is delivery affected by cross state?","Cross-state", "Same-state")
])

cross_tbl


,Metric,Cross-state N,Same-state N,Cross-state median,Same-state median,Median difference,Relative difference %,p-value,Effect size
0,Is delivery affected by cross state?,61774,34695,9.0,3.0,6.0,200.0,0.0,-0.679982


In [205]:

cross_state_pct = (
    cross_tbl["Cross-state N"].iloc[0]
    / (cross_tbl["Cross-state N"].iloc[0] + cross_tbl["Same-state N"].iloc[0])
    * 100
)

print(f"Cross-state % orders: {cross_state_pct:.2f}%")

print(f"Last Mile Median difference > Material Threshold?: {abs(cross_tbl["Median difference"].iloc[0]) >= MATERIAL_LAST_MILE_DAYS}")

Cross-state % orders: 64.04%
Last Mile Median difference > Material Threshold?: True


**Result:** Median last mile is **9.0 days cross-state (n=61,774)** vs **3.0 days same-state
(n=34,695)** — a **6-day difference**, far exceeding the predefined 1-day materiality threshold. The distributional effect is also large (effect size ≈ -0.68).

**Insight:** Cross-state delivery is associated with a substantial increase in last-mile duration, and it applies to **64% of all orders** in this comparison population — the seller base is concentrated in the south-east while demand is national.(NB01 - S10)
This makes geography an important factor to investigate in the last-mile stage. However, this analysis does not establish why cross-state deliveries take longer; geographic distance, routing, regional coverage, and other factors may contribute.

**Recommendation:** Use cross-state status as a segmentation variable for delivery-performance monitoring.

The next investigation should break the 6-day gap down by seller state → customer state routes and geography to identify where the longest delays occur. If specific regions or routes consistently show higher last-mile duration, those corridors can then be evaluated for carrier coverage, fulfilment capacity, or regional seller-network changes.

### B4. Where is reliability worst?

**Question**: Which customer states have lower on-time performance, and where does poor reliability affect the largest order volume?

**Population**: Delivered orders with known on-time outcome and last-mile duration; states with ≥500 orders.

In [221]:
delivered.shape

(96470, 20)

In [230]:
state_data = delivered.dropna(subset=["is_delivered_on_time", "carrier_to_customer_days", "is_cross_state"]).copy()

by_state = (state_data.groupby("customer_state")
                     .agg(orders=("is_delivered_on_time", "size"),
                          on_time_orders=("is_delivered_on_time", "sum"),
                          late_orders=("is_delivered_on_time", lambda x:(~x).sum()),
                          on_time_pct=("is_delivered_on_time", "mean"),
                          median_last_mile_days=("carrier_to_customer_days", "median")))
by_state["on_time_pct"] = (by_state["on_time_pct"] * 100).round(1)
by_state = by_state[by_state["orders"] >= 500].sort_values("on_time_pct", ascending=False)
by_state[by_state["on_time_pct"]>=90]

,orders,on_time_orders,late_orders,on_time_pct,median_last_mile_days
customer_state,,,,,
PR,4921,4675,246,95.0,7.0
MG,11351,10713,638,94.4,7.0
SP,40498,38112,2386,94.1,4.0
MT,886,825,61,93.1,13.0
DF,2077,1932,145,93.0,8.0
RS,5344,4962,382,92.9,10.0
GO,1956,1797,159,91.9,10.0
SC,3547,3201,346,90.2,9.0


In [231]:
by_state[by_state["on_time_pct"]<90]

,orders,on_time_orders,late_orders,on_time_pct,median_last_mile_days
customer_state,,,,,
PB,515,459,56,89.1,14.0
PE,1592,1419,173,89.1,12.0
MS,700,619,81,88.4,11.0
ES,1996,1752,244,87.8,9.0
PA,946,829,117,87.6,17.0
RJ,12351,10686,1665,86.5,8.0
BA,3254,2797,457,86.0,13.0
CE,1281,1085,196,84.7,14.0
MA,720,579,141,80.4,15.0


In [232]:
state_mix = state_data.groupby("customer_state").agg(
    orders = ("is_cross_state", "size"),
    cross_state_orders = ("is_cross_state", "sum")
)

state_mix["cross_state_pct"] = (state_mix["cross_state_orders"] / state_mix["orders"] * 100).round(1)
state_mix["same_state_pct"] = (100 - state_mix["cross_state_pct"]).round(1)

by_state = by_state.join(state_mix[["cross_state_pct", "same_state_pct"]])

by_state

,orders,on_time_orders,late_orders,on_time_pct,median_last_mile_days,cross_state_pct,same_state_pct
customer_state,,,,,,,
PR,4921,4675,246,95.0,7.0,85.5,14.5
MG,11351,10713,638,94.4,7.0,86.6,13.4
SP,40498,38112,2386,94.1,4.0,24.1,75.9
MT,886,825,61,93.1,13.0,99.3,0.7
DF,2077,1932,145,93.0,8.0,97.6,2.4
RS,5344,4962,382,92.9,10.0,94.6,5.4
GO,1956,1797,159,91.9,10.0,98.4,1.6
SC,3547,3201,346,90.2,9.0,92.8,7.2
PB,515,459,56,89.1,14.0,99.8,0.2


**Insight:** **Delivery reliability varies materially by customer state, and the business impact depends on both rate and volume.**.

MA records the lowest rate among states meeting the 500-order threshold, while RJ combines a below-90% on-time rate with a much larger order base. **RJ therefore represents the largest concentration of delivered-order exposure among the underperforming states.** 9 states with ≥500 orders fall below 90% on-time; **RJ alone accounts for 12,351 of those delivered orders against 11,004 across the other eight combined**, and produces
**1,665 late orders**. RJ is where weak reliability meets high volume.

The cross-state mix provides important context because B3 showed a substantial last-mile duration difference between cross-state and same-state deliveries. However, state-level reliability cannot be attributed to cross-state exposure alone without further route-level analysis.

**Recommendation:** Prioritize RJ for deeper operational investigation because it combines below-90% on-time performance with the largest order volume among underperforming states.

For MA, CE, BA and other low-reliability states, investigate the underlying seller-state → customer-state routes, cross-state mix, and last-mile duration before deciding between operational intervention and estimate recalibration.

### B5 — Revenue exposure from late delivery

Question: What share of delivered-order revenue is associated with late deliveries?

In [254]:
late_delivered = delivered.loc[~delivered["is_delivered_on_time"], "order_total_revenue"]
on_time_delivered = delivered.loc[delivered["is_delivered_on_time"], "order_total_revenue"]

late_median = late_delivered.median()
on_time_median = on_time_delivered.median()

total_rev = delivered["order_total_revenue"]

relative_diff = (late_median-on_time_median)/on_time_median

result = pd.DataFrame([
    ["Late orders", len(late_delivered)],
    ["Revenue from late orders", f"R${late_delivered.sum():,.0f}"],
    ["Late-order revenue share%", f"{(late_delivered.sum()/total_rev.sum())*100:.2f}%"],
    ["Median order value — late", f"R${late_median:.2f}"],
    ["Median order value - on time", f"R${on_time_median:.2f}"],
    ["Relative difference", f"{relative_diff:.3f}"],
    ["Material Threshold?", f"{ abs(relative_diff) > MATERIAL_MONETARY_RELATIVE}"]
], columns = ["Measure", "Value"])

result

,Measure,Value
0,Late orders,7826
1,Revenue from late orders,"R$1,158,921"
2,Late-order revenue share%,8.77%
3,Median order value — late,R$89.90
4,Median order value - on time,R$85.50
5,Relative difference,0.051
6,Material Threshold?,False


**Insight**: Late deliveries account for 8.77% of delivered-order revenue, slightly above their 8.1% share of orders (late).

However, the typical order value is similar between late and on-time deliveries: R$89.90 vs R$85.50, a difference of approximately 5.1%, below the project's 10% monetary materiality threshold.

Therefore, the commercial exposure from late delivery is meaningful primarily because of the volume of affected orders, rather than because late orders are materially higher-value orders.

**Recommendation**: Treat the 8.77% revenue exposure as a business-impact metric alongside delivery reliability, rather than as lost or at-risk revenue.

Operational improvement should focus on reducing the volume of late deliveries, particularly in the high-volume states and routes identified in B4, where reliability problems affect more orders.

For prioritization, combine: late-delivery rate × order volume × revenue exposure, rather than targeting high-value orders based on order value alone.

### B6 — Does delivery performance vary by category?

**Question**: Do delivery reliability and last-mile duration differ across product categories?

Categories with at least 100 orders are shown individually to avoid interpreting very small category samples.

In [276]:
category_data = delivered.dropna(subset=[
    "primary_category", "is_delivered_on_time", "carrier_to_customer_days"
    ]).copy()

f = category_data.groupby("primary_category").agg(
    orders=("is_delivered_on_time", "size"),
    on_time_pct=("is_delivered_on_time","mean"),
    median_last_mile=("carrier_to_customer_days", "median")
)

f["on_time_pct"] = (f["on_time_pct"]*100).round(1)

f = f[f["orders"]>=100].sort_values("on_time_pct", ascending=False)

In [275]:
category_summary = f.copy()

top_categories = category_summary.head(5)
bottom_categories = category_summary.tail(5).sort_values("on_time_pct")

display(top_categories)
display(bottom_categories)

,orders,on_time_pct,median_last_mile
primary_category,,,
climatizacao,245,95.9,6.0
construcao_ferramentas_seguranca,154,95.5,6.0
moveis_cozinha_area_de_servico_jantar_e_jardim,237,95.4,6.0
fashion_roupa_masculina,106,95.3,6.5
market_place,268,95.1,6.0


,orders,on_time_pct,median_last_mile
primary_category,,,
audio,344,86.9,7.0
fashion_underwear_e_moda_praia,117,87.2,7.0
livros_tecnicos,254,89.0,6.0
casa_conforto,367,89.6,8.0
alimentos,436,89.9,5.0


**Insight:** Delivery reliability varies materially across product categories. Among categories with at least 100 orders, on-time performance ranges from 86.9% to 95.9%, a 9.0 pp gap. However, category-level on-time performance does not move directly with median last-mile duration, suggesting that promise adherence and actual delivery duration capture different aspects of delivery performance.

**Recommendation:** Use product category as a diagnostic segmentation rather than treating it as a standalone cause of delivery problems. Investigate low-performing categories by cross-state mix, seller geography, and delivery routes to determine whether the observed differences persist after accounting for operational context.

In [277]:
delivered.columns

Index(['order_id', 'order_status', 'is_delivered', 'customer_key',
       'customer_state', 'customer_city', 'primary_seller_key',
       'primary_seller_state', 'is_cross_state', 'order_purchase_date_key',
       'approval_to_carrier_hours', 'carrier_to_customer_days',
       'carrier_to_customer_hours', 'is_delivered_on_time',
       'delivery_estimate_error_days', 'order_total_revenue',
       'order_total_freight', 'freight_to_revenue_ratio', 'primary_category',
       'has_delivery_timestamp_anomaly'],
      dtype='str')

## Part C. Customer Experience & Satisfaction

This section tests an assumption rather than confirming one: **do operational failures actually
reach the customer?**

`review_score` is ordinal (NB01 S7), so medians and distributions are reported; means appear only
as a familiar reference. A material gap is ≥0.30 stars.

**Population:** 97,917 reviewed orders (99.2% of orders).

In [282]:
print(reviewed.shape)
print(reviewed_delivered.shape)
print(experience.shape)

(97917, 16)
(95830, 16)
(98666, 16)


In [280]:
experience.columns

Index(['order_id', 'order_status', 'customer_key', 'customer_state',
       'is_cross_state', 'order_purchase_date_key', 'review_score',
       'has_review', 'is_negative_review', 'has_fulfillment_anomaly',
       'anomaly_severity_tier', 'is_delivered_on_time',
       'delivery_estimate_error_days', 'lateness_bucket',
       'order_total_revenue', 'primary_category'],
      dtype='str')

### C1. Do internal data anomalies relate to customer satisfaction?
     — Tests the assumption that our data-quality defects are also CX problems.
     — Population: all reviewed orders.
**Note:** the severity tiers `missing_data` (15 orders) and `contradiction` (6) are too small to
interpret, so the comparison is **none vs. timing_or_value**, as established in NB01.

In [284]:
anom = reviewed[reviewed["has_fulfillment_anomaly"]]["review_score"]
none = reviewed[~reviewed["has_fulfillment_anomaly"]]["review_score"]

anom_tbl = pd.DataFrame([compare_median_value(anom, none, "Anomaly vs Review", "Anomaly", "No anomaly")])

anom_tbl

,Metric,Anomaly N,No anomaly N,Anomaly median,No anomaly median,Median difference,Relative difference %,p-value,Effect size
0,Anomaly vs Review,1405,96512,5.0,5.0,0.0,0.0,0.000003,-0.065


In [290]:
detractor_gap = (anom <= 2).mean() * 100 - (none <= 2).mean() * 100
c1 = pd.DataFrame([
    ["Anomaly",    len(anom), anom.median(), round((anom <= 2).mean() * 100, 1)],
    ["No anomaly", len(none), none.median(), round((none <= 2).mean() * 100, 1)],
], columns=["Group", "n", "Median review", "% 1–2 star"])

print(f"Median difference: {anom_tbl.iloc[0]["Median difference"]} stars")
print(f"Detractor-rate difference: {detractor_gap:+.1f}pp (material at {MATERIAL_RATE_DIFF_PP}pp) "
      f"| effect={anom_tbl.iloc[0]["Effect size"]:.3f} | p={anom_tbl.iloc[0]["p-value"]:.3g}")
c1

Median difference: 0.0 stars
Detractor-rate difference: -3.3pp (material at 2.0pp) | effect=-0.065 | p=2.59e-06


,Group,n,Median review,% 1–2 star
0,Anomaly,1405,5.0,11.0
1,No anomaly,96512,5.0,14.2


**Result:** Median review score is **5 in both groups** — no difference. Anomaly orders have a
**11.0%** detractor rate (1–2 star) against **14.2%** without, a difference of **−3.3pp** in the
*favourable* direction, with a negligible effect size (0.064).

**Insight:** The hypothesis is not supported. Fulfilment anomalies are timestamp-ordering defects
in our own records — an approval logged after carrier pickup, a missing lifecycle date — and the
customer never sees them. The small favourable difference is not evidence that anomalies help;
with an effect size near zero it is better read as no meaningful relationship. 

**Recommendation:** Treat fulfillment anomalies primarily as a data-quality monitoring issue, rather than a customer-experience signal. They should not be used directly as evidence of poor seller or customer service performance.

### C2.  Does late delivery hurt customer satisfaction — and at what point?
    — Tests whether the customer-visible failure (lateness) produces measurable dissatisfaction.
    — Population: reviewed + delivered orders, split by lateness_bucket.

**Question:** Does lateness — the failure customers actually experience — affect satisfaction?
**Population:** 95,830 reviewed and delivered orders (`not delivered` excluded).

In [292]:
reviewed_delivered[["review_score", "is_negative_review", "lateness_bucket"]].head()

,review_score,is_negative_review,lateness_bucket
1,5,False,early
2,5,False,early
3,5,False,early
4,5,False,early
5,5,False,early


In [407]:
late = reviewed_delivered.groupby("lateness_bucket").agg(
    total_reviewed_orders=("review_score", "size"),
    median_review=("review_score", "median"),
    review_1_2_no = ("is_negative_review", "sum"),         ## is_negative_review contains reviews with <=2 score
    pct_1_2_star=("is_negative_review", lambda x: (x.mean()*100).round(1))
)
late

,total_reviewed_orders,median_review,review_1_2_no,pct_1_2_star
lateness_bucket,,,,
early,88168,5.0,8133,9.2
on_time,1280,5.0,159,12.4
slightly_late,1852,4.0,595,32.1
very_late,4530,1.0,3387,74.8


In [307]:
very_late = reviewed_delivered.loc[reviewed_delivered["lateness_bucket"] == "very_late", "review_score"]
early     = reviewed_delivered.loc[reviewed_delivered["lateness_bucket"] == "early", "review_score"]

late_tbl = compare_median_value(very_late, early,"check", "Very late", "Early")

detractor_gap_late = (very_late <= 2).mean() * 100 - (early <= 2).mean() * 100
print(f"very_late vs early | median {very_late.median():.0f} vs {early.median():.0f} "
      f"| detractor {detractor_gap_late:+.1f}pp (material at {MATERIAL_RATE_DIFF_PP}pp) "
      f"| effect={late_tbl["Effect size"]:+.3f} | p={late_tbl["p-value"]:.3g}")


very_late vs early | median 1 vs 5 | detractor +65.5pp (material at 2.0pp) | effect=+0.750 | p=0


**Insight:** **Customer satisfaction deteriorates sharply as delivery lateness increases.**
The deterioration appears before extreme delays: the median review falls from 5.0 to 4.0 for slightly late orders, while the 1–2 star rate rises from 9.2% to 32.1%. For very-late orders, the deterioration becomes severe, with 74.8% receiving 1–2 stars and a median review of 1.0.

This suggests that delivery reliability is strongly associated with customer satisfaction, and that the customer-experience impact is not limited to the most extreme delays.

**Recommendation:** **Prioritize preventing delivery delays, not just reducing extreme delays.**

Because even slightly late orders show a substantial increase in negative reviews, operational monitoring should track lateness severity, rather than relying only on the overall on-time rate.

Use the findings from Part B to identify where intervention should be focused — particularly high-volume states, cross-state deliveries, and routes with longer last-mile durations — and monitor whether reducing lateness in those segments corresponds with improved customer satisfaction.

In [421]:
a = experience[experience["order_status"] == 'delivered'].copy()

ans = a.groupby("lateness_bucket").agg(
    total_revenue=("order_total_revenue", "sum"),
    total_percentage=("order_total_revenue", lambda x: (x.sum()/a["order_total_revenue"].sum()*100).round(1))
)
ans

,total_revenue,total_percentage
lateness_bucket,,
early,12061328.42,91.2
not delivered,1249.18,0.0
on_time,172996.17,1.3
slightly_late,250856.63,1.9
very_late,735067.71,5.6


## N7. Tableau Export

Five aggregated extracts, one per planned view. Pre-aggregated rather than raw, so each Tableau
worksheet is a direct read with no recalculation of business logic.

In [308]:
sellers.columns

Index(['seller_key', 'seller_id', 'seller_city', 'seller_state',
       'seller_tenure_bucket', 'seller_volume_tier', 'total_orders',
       'total_items', 'total_revenue', 'total_freight', 'avg_order_value',
       'avg_freight_to_price_ratio', 'primary_category',
       'distinct_categories_sold', 'review_count', 'avg_review_score',
       'pct_negative_reviews', 'delivered_order_count', 'late_delivery_count',
       'late_delivery_rate', 'shipping_anomaly_count',
       'item_level_anomaly_rate', 'order_level_anomalous_order_count',
       'order_level_anomaly_rate', 'delivered_missing_approval_orders',
       'delivered_missing_multiple_dates_orders',
       'carrier_before_approval_orders', 'delivery_before_carrier_orders',
       'canceled_with_delivery_date_orders',
       'delivered_missing_final_date_orders',
       'delivered_missing_carrier_anomaly_orders',
       'has_sufficient_sample_size', 'has_sufficient_delivery_sample_size',
       'is_reliable_seller'],
      dtype='

In [312]:
print(reliable_sellers.shape)
print(sellers.shape)

(1766, 34)
(3095, 34)


### 1. sellers table

In [355]:
seller_risk = sellers[[
    "seller_key", "seller_city", "seller_state", "seller_tenure_bucket", "seller_volume_tier", 
    'total_orders', 'total_items', 'total_revenue', 'total_freight', 'avg_order_value',
    'avg_freight_to_price_ratio', 'primary_category', 'distinct_categories_sold', "review_count",
    'avg_review_score', "pct_negative_reviews", "delivered_order_count", "order_level_anomaly_rate", 
    "late_delivery_rate", "is_reliable_seller"]].copy()

seller_risk.to_csv(EXPORT_DIR / "seller_risk.csv", index=False)

In [315]:
print(experience.columns)
print(reviewed.shape)
print(reviewed_delivered.shape)

Index(['order_id', 'order_status', 'customer_key', 'customer_state',
       'is_cross_state', 'order_purchase_date_key', 'review_score',
       'has_review', 'is_negative_review', 'has_fulfillment_anomaly',
       'anomaly_severity_tier', 'is_delivered_on_time',
       'delivery_estimate_error_days', 'lateness_bucket',
       'order_total_revenue', 'primary_category'],
      dtype='str')
(97917, 16)
(95830, 16)


### 2. experience table

In [360]:
customer_experience = experience[[
    'order_id','order_status', 'customer_key', 'customer_state',
    'is_cross_state', 'order_purchase_date_key', 'review_score',
    'has_review', 'is_negative_review', 'has_fulfillment_anomaly',
    'anomaly_severity_tier', 'is_delivered_on_time',
    'delivery_estimate_error_days', 'lateness_bucket',
    'order_total_revenue', 'primary_category'
]].copy()

customer_experience.to_csv(EXPORT_DIR / "customer_experience.csv", index=False)

### 3. Delivery table

In [350]:
delivery.columns

Index(['order_id', 'order_status', 'is_delivered', 'customer_key',
       'customer_state', 'customer_city', 'primary_seller_key',
       'primary_seller_state', 'is_cross_state', 'order_purchase_date_key',
       'approval_to_carrier_hours', 'carrier_to_customer_days',
       'carrier_to_customer_hours', 'is_delivered_on_time',
       'delivery_estimate_error_days', 'order_total_revenue',
       'order_total_freight', 'freight_to_revenue_ratio', 'primary_category',
       'has_delivery_timestamp_anomaly'],
      dtype='str')

In [361]:
delivery_experience = delivery[[
     'order_id','order_status', 'is_delivered', 'customer_key',
     'customer_state', 'customer_city', 'primary_seller_key',
     'primary_seller_state', 'is_cross_state', 'order_purchase_date_key',
     'approval_to_carrier_hours', 'carrier_to_customer_days',
     'carrier_to_customer_hours', 'is_delivered_on_time',
     'delivery_estimate_error_days', 'order_total_revenue',
     'order_total_freight', 'freight_to_revenue_ratio', 'primary_category'
]].copy()

delivery_experience.to_csv(EXPORT_DIR / "delivery_experience.csv", index = False)

### 4. Customer rfm table

In [352]:
rfm.columns

Index(['customer_key', 'customer_unique_id', 'customer_state',
       'first_purchase_category', 'first_purchase_month', 'recency_days',
       'recency_score', 'frequency', 'is_repeat_buyer', 'monetary',
       'monetary_score', 'avg_order_value', 'first_order_date',
       'last_order_date', 'customer_lifespan_days', 'acq_month'],
      dtype='str')

In [358]:
customer_rfm = rfm[[
    'customer_key', 'customer_unique_id', 'customer_state',
    'first_purchase_category', 'first_purchase_month', 'recency_days',
    'recency_score', 'frequency', 'is_repeat_buyer', 'monetary',
    'monetary_score', 'avg_order_value', 'first_order_date',
    'last_order_date', 'customer_lifespan_days'
]].copy()

customer_rfm.to_csv(EXPORT_DIR / "customer_rfm.csv", index = False)

In [362]:
# ============================================================
# VALIDATION
# ============================================================

print("TABLEAU EXPORTS COMPLETE")
print("-" * 60)

print(f"seller_risk.csv          : {len(seller_risk):,} rows")
print(f"delivery_experience.csv : {len(delivery_experience):,} rows")
print(f"customer_experience.csv: {len(customer_experience):,} rows")
print(f"customer_rfm.csv    : {len(customer_rfm):,} rows")

print("-" * 60)

print("Seller grain       :", seller_risk["seller_key"].nunique(), "unique sellers")
print("Delivery grain     :", delivery_experience["order_id"].nunique(), "unique orders")
print("Experience grain   :", customer_experience["order_id"].nunique(), "unique orders")
print("Customer grain     :", customer_rfm["customer_key"].nunique(), "unique customers")

TABLEAU EXPORTS COMPLETE
------------------------------------------------------------
seller_risk.csv          : 3,095 rows
delivery_experience.csv : 98,666 rows
customer_experience.csv: 98,666 rows
customer_rfm.csv    : 94,989 rows
------------------------------------------------------------
Seller grain       : 3095 unique sellers
Delivery grain     : 98666 unique orders
Experience grain   : 98666 unique orders
Customer grain     : 94989 unique customers


In [369]:
rfm.nunique()

customer_key               94989
customer_unique_id         94989
customer_state                27
first_purchase_category       74
first_purchase_month          23
recency_days                 613
recency_score                  5
frequency                      9
is_repeat_buyer                2
monetary                   31654
monetary_score                 5
avg_order_value            31816
first_order_date             613
last_order_date              613
customer_lifespan_days       422
acq_month                     23
dtype: int64

In [372]:
delivery.nunique()

order_id                          98666
order_status                          7
is_delivered                          2
customer_key                      95420
customer_state                       27
customer_city                      4094
primary_seller_key                 3085
primary_seller_state                 23
is_cross_state                        2
order_purchase_date_key             616
approval_to_carrier_hours           918
carrier_to_customer_days            144
carrier_to_customer_hours           144
is_delivered_on_time                  2
delivery_estimate_error_days        198
order_total_revenue                7558
order_total_freight                7560
freight_to_revenue_ratio           9292
primary_category                     74
has_delivery_timestamp_anomaly        2
dtype: int64

In [373]:
delivery['is_delivered_on_time'].value_counts()

is_delivered_on_time
True     88649
False     7827
Name: count, dtype: Int64